# Exercise: Your First dapi Job

One calculation, every dapi step typed by you. A 5%-damped oscillator is shaken exactly at its natural frequency, and one job on Stampede3 integrates the motion and reports the dynamic amplification, which must land near $1/(2\xi) = 10$.

![A mass on a spring and damper, forced by a sine load, beside the amplification curve, with marked points.](resonance_schematic.png)

The oscillator obeys

$$\ddot{x} + 2\xi\omega_n\dot{x} + \omega_n^2 x = \frac{F_0}{m}\sin(\Omega t)$$

and this exercise computes the single point at $\Omega = \omega_n$, the top of the curve. Each TODO is one dapi call; the hints hold the exact call, and the [solution notebook](first-dapi-job.ipynb) shows the whole thing executed.

In [ ]:
# Install

**Restart the kernel once after the install**, then run from the next cell.

## TODO 1. Authenticate

Create the client. It reads your DesignSafe credentials from the environment or a `.env` file, or prompts for them, and every later call goes through it.

<details><summary>Hint</summary>

`from dapi import DSClient` then `ds = DSClient()`.
</details>

In [ ]:
...  # TODO

## The analysis (given)

The script integrates the oscillator at resonance and writes one number to `result.json`. The folder it lands in becomes the job's input directory.

In [ ]:
from pathlib import Path

input_dir = Path.cwd() / "oscillator_job"
input_dir.mkdir(exist_ok=True)

script = """\
# A 5%-damped oscillator shaken at its natural frequency.

import json

import numpy as np

xi = 0.05
wn = 1.0
w = 1.0 * wn                       # forcing at resonance

dt = 0.002
n = int(80 * 2 * np.pi / wn / dt)  # 80 natural periods
x = np.zeros(n)
t = np.arange(n) * dt
for i in range(1, n - 1):
    a = np.sin(w * t[i]) - 2 * xi * wn * (x[i] - x[i - 1]) / dt - wn**2 * x[i]
    x[i + 1] = 2 * x[i] - x[i - 1] + a * dt**2

steady = x[int(0.75 * n):]
amplification = float(np.max(np.abs(steady)) * wn**2)

json.dump({"ratio": w / wn, "amplification": amplification}, open("result.json", "w"))
print(f"amplification at resonance: {amplification:.3f}")
"""

(input_dir / "oscillator.py").write_text(script)
print(f"Wrote {input_dir}/oscillator.py")

## TODO 2. Point Tapis at the folder

Jobs read inputs from DesignSafe storage, not from your disk, so the folder needs a `tapis://` address. On JupyterHub it translates directly; from anywhere else, upload it once.

<details><summary>Hint</summary>

`ds.files.to_uri(str(input_dir))` on JupyterHub; it raises `ValueError` elsewhere, so catch that and use `ds.jobs.prepare_inputs("python-s3", str(input_dir))`, translating its `staged_dir` instead.
</details>

In [ ]:
input_uri = ...  # TODO
print("Input URI:", input_uri)

## TODO 3. Generate the job request

Build the complete request from the `python-s3` app definition, the script as the entry point, one core of one node, ten minutes, `skx-dev`. Print the dict; this is exactly what Tapis will receive.

<details><summary>Hint</summary>

`ds.jobs.generate(app_id="python-s3", input_dir_uri=input_uri, script_filename="oscillator.py", node_count=1, cores_per_node=1, max_minutes=10, queue="skx-dev", allocation=...)`.
</details>

In [ ]:
import json  # noqa: F401  (used below)

allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

job = ...  # TODO
print(json.dumps(job, indent=2, default=str))

## TODO 4. Submit and monitor

Hand the request to Tapis, then poll until the job reaches a terminal state, watching it move through staging, queueing, running, and archiving.

<details><summary>Hint</summary>

`submitted = ds.jobs.submit(job)` then `final = submitted.monitor(interval=15)`, and `ds.jobs.interpret_status(final, submitted.uuid)` explains the outcome. `submitted.print_runtime_summary()` splits the wall clock by stage.
</details>

In [ ]:
submitted = ...  # TODO
final = ...  # TODO
# TODO: interpret the status and print the runtime summary

## TODO 5. Read the archive

Tapis copied everything the job produced to your storage. List the archived input folder, then read `result.json` back and check the physics.

<details><summary>Hint</summary>

`submitted.archive_uri` names the archive; list `archive_uri + "/inputDirectory"` with `ds.files.list`. `submitted.get_output_content("inputDirectory/result.json")` returns the file's text, `json.loads` it. Expect amplification between 9 and 11.
</details>

In [ ]:
# TODO: list the archived inputDirectory

result = ...  # TODO
assert 9 < result["amplification"] < 11
print(f"D at resonance = {result['amplification']:.2f}")

## Going further

Change the forcing ratio in `oscillator.py` and resubmit; each run computes another point of the curve. The [PyLauncher sweep exercise](../pylauncher/pylauncher_sweep_exercise.ipynb) runs all 25 points inside a single job.